In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [5]:
import pandas as pd

from src.silver.reader import read_bronze_table

In [6]:
df_alunos = read_bronze_table("alunos")
df_municipio = read_bronze_table("municipio")
df_meta_municipio = read_bronze_table("meta_municipio")

In [7]:
def find_relationship_violations(
    df: pd.DataFrame,
    reference_df: pd.DataFrame,
    columns: list[str],
    reference_columns: list[str],
) -> pd.DataFrame:
    left_keys = df[columns].dropna().drop_duplicates()

    right_keys = (
        reference_df[reference_columns]
        .dropna()
        .drop_duplicates()
    )

    right_keys.columns = columns

    validation = left_keys.merge(
        right_keys,
        on=columns,
        how="left",
        indicator=True,
    )

    return validation[validation["_merge"] == "left_only"].drop(columns=["_merge"])

In [8]:
viol_alunos_municipio = find_relationship_violations(
    df=df_alunos,
    reference_df=df_municipio,
    columns=["ano", "id_municipio"],
    reference_columns=["ano", "id_municipio"],
)

viol_alunos_municipio

,ano,id_municipio
47,2024,4203204
182,2024,4202008
461,2023,5219308
9680,2024,4305371
10265,2023,4128625


In [9]:
viol_alunos_municipio["ano"].value_counts(dropna=False)

ano
2024    3
2023    2
Name: count, dtype: Int64

In [10]:
alunos_afetados = df_alunos.merge(
    viol_alunos_municipio,
    on=["ano", "id_municipio"],
    how="inner",
)

alunos_afetados[
    ["ano", "id_municipio", "id_municipio_nome", "id_escola", "id_aluno", "rede"]
].head(20)

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,rede
0,2024,4203204,Camboriú,60035994,42016633,Municipal
1,2024,4202008,Balneário Camboriú,60035967,42003898,Municipal
2,2024,4202008,Balneário Camboriú,60035278,42004251,Municipal
3,2024,4202008,Balneário Camboriú,60035278,42004172,Municipal
4,2023,5219308,Santa Helena de Goiás,60042125,52064217,Municipal
5,2024,4203204,Camboriú,60035282,42016928,Municipal
6,2023,5219308,Santa Helena de Goiás,60042125,52064205,Municipal
7,2023,5219308,Santa Helena de Goiás,60042125,52064193,Municipal
8,2024,4202008,Balneário Camboriú,60036165,42003211,Municipal
9,2023,5219308,Santa Helena de Goiás,60042125,52064214,Municipal


In [11]:
viol_municipio_meta = find_relationship_violations(
    df=df_municipio,
    reference_df=df_meta_municipio,
    columns=["ano", "id_municipio"],
    reference_columns=["ano", "id_municipio"],
)

viol_municipio_meta

,ano,id_municipio
51,2023,2413102
93,2023,3100609
171,2023,4207106
176,2023,4211306
185,2023,4302154
...,...,...
10781,2024,3136520
10926,2024,4323754
10945,2024,3126505
10997,2024,3503950


In [12]:
viol_municipio_meta["ano"].value_counts(dropna=False)

ano
2023    195
2024    164
Name: count, dtype: Int64

In [13]:
municipios_afetados = df_municipio.merge(
    viol_municipio_meta,
    on=["ano", "id_municipio"],
    how="inner",
)

municipios_afetados[
    ["ano", "id_municipio", "id_municipio_nome", "rede", "taxa_alfabetizacao"]
].head(30)

,ano,id_municipio,id_municipio_nome,rede,taxa_alfabetizacao
0,2023,2413102,Senador Elói de Souza,Pública (Estadual e Municipal),50.36
1,2023,3100609,Água Boa,Estadual,66.70
2,2023,4207106,Ilhota,Estadual,38.95
3,2023,4211306,Navegantes,Municipal,54.66
4,2023,4302154,Boa Vista das Missões,Municipal,77.20
5,2023,1300904,Canutama,Estadual,84.63
6,2023,2412708,São Pedro,Estadual,42.86
7,2023,2904753,Buritirama,Municipal,13.04
8,2023,3123502,Douradoquara,Pública (Estadual e Municipal),66.67
9,2023,3166501,Serra Azul de Minas,Estadual,55.00


In [14]:
pd.DataFrame({
    "tabela": ["alunos", "municipio", "municipio", "meta_municipio"],
    "coluna": ["id_municipio", "id_municipio", "id_municipio", "id_municipio"],
    "dtype": [
        df_alunos["id_municipio"].dtype,
        df_municipio["id_municipio"].dtype,
        df_municipio["id_municipio"].dtype,
        df_meta_municipio["id_municipio"].dtype,
    ],
})

,tabela,coluna,dtype
0,alunos,id_municipio,object
1,municipio,id_municipio,object
2,municipio,id_municipio,object
3,meta_municipio,id_municipio,object


In [15]:
pd.DataFrame({
    "tabela": ["alunos", "municipio", "meta_municipio"],
    "min_len": [
        df_alunos["id_municipio"].astype(str).str.len().min(),
        df_municipio["id_municipio"].astype(str).str.len().min(),
        df_meta_municipio["id_municipio"].astype(str).str.len().min(),
    ],
    "max_len": [
        df_alunos["id_municipio"].astype(str).str.len().max(),
        df_municipio["id_municipio"].astype(str).str.len().max(),
        df_meta_municipio["id_municipio"].astype(str).str.len().max(),
    ],
})

,tabela,min_len,max_len
0,alunos,7,7
1,municipio,7,7
2,meta_municipio,7,7


In [16]:
find_relationship_violations(
    df=df_alunos,
    reference_df=df_municipio,
    columns=["id_municipio"],
    reference_columns=["id_municipio"],
)


,id_municipio
412,5219308


In [17]:
find_relationship_violations(
    df=df_municipio,
    reference_df=df_meta_municipio,
    columns=["id_municipio"],
    reference_columns=["id_municipio"],
)

,id_municipio
51,2413102
93,3100609
171,4207106
176,4211306
185,4302154
...,...
5463,3138302
5487,2918456
5520,4300471
5531,5300108


In [18]:
viol_municipio_meta_sem_ano = find_relationship_violations(
    df=df_municipio,
    reference_df=df_meta_municipio,
    columns=["id_municipio"],
    reference_columns=["id_municipio"],
)

municipios_sem_meta = df_municipio.merge(
    viol_municipio_meta_sem_ano,
    on="id_municipio",
    how="inner",
)

municipios_sem_meta["rede"].value_counts()

rede
Pública (Estadual e Municipal)                    341
Municipal                                         242
Estadual                                          237
Total (Federal, Estadual, Municipal e Privada)      9
Name: count, dtype: int64